# ArkIve — Construção do índice RAG (FAISS)

Gera/atualiza o índice vetorial usado pelo Motor ArkIve para buscar as **3 doenças mais relevantes** a partir da transcrição da consulta.

| Item | Escolha |
|---|---|
| Vector store | FAISS `IndexIDMap2(IndexFlatIP)` — busca exata por similaridade de cosseno |
| Embeddings | `multilingual-e5-small` em ONNX int8 (118 MB) via `fastembed` — sem torch, roda no Render free (512 MB) |
| Idioma | documentos em inglês, consulta em PT-BR (modelo multilíngue) |

**O backend precisa usar exatamente o mesmo modelo** (`MODEL_NAME`/`MODEL_FILE`) e a mesma versão do `fastembed` para vetorizar a consulta — senão os vetores ficam incompatíveis com o índice.

## Como usar

**Primeira vez (X arquivos):**
1. Zipe o **conteúdo** de `RAG_REFERENCES` (pastas `AAZV_MD/`, `WOAH_MD/`... na raiz do zip) → `docs.zip`.
   No Windows use Explorer ("Enviar para > Pasta compactada") ou 7-Zip — `Compress-Archive` do PowerShell 5.1 grava `\` nos caminhos.
2. Rode todas as células. Na célula de upload envie **só** `docs.zip`.
3. Baixe `rag_index.zip` no final e extraia em `rag/index/` no repositório.

**Adicionar mais Y arquivos (incremental):**
1. Zipe **somente os novos** `.md` → `docs.zip` (mesma estrutura de pastas; a chave de cada documento é o caminho relativo dentro do zip).
2. Rode tudo. No upload envie `rag_index.zip` (índice atual, de `rag/index/`) **e** `docs.zip`.
3. Só os arquivos novos ou alterados são vetorizados. Baixe o novo `rag_index.zip` e substitua `rag/index/`.

Arquivo com mesmo caminho e conteúdo diferente é **substituído** (chunks antigos removidos). Arquivos `.json` do Docling são ignorados — o `.md` já tem o mesmo conteúdo.

## Limitações conhecidas
- Fontes: WOAH, AAZV, CFSPH, CAPC, ABCD, ESCCAP (inglês). WOAH quase não traz tratamento; cão/gato vêm de CAPC/ABCD/ESCCAP.
- Mesma doença pode existir em várias fontes (ex.: raiva em WOAH/CFSPH/AAZV) e ocupar mais de uma vaga do top-3. Guias ESCCAP `GL1..GL9` entram como documento (diagnóstico/controle por grupo de parasita); o título exibido é o do guia.
- Linhas de tabela markdown (`|...`) são descartadas — tabelas de risco/drogas (ESCCAP, AAZV) não entram no índice.
- Incremental: arquivo novo com mesmo nome de um já indexado em outra pasta (ex.: `Fleas.md`) dispara o alerta de pasta raiz da célula 7.
- `score` é similaridade de cosseno (0–1), **não** probabilidade clínica.
- e5-small lê até 512 tokens por texto: transcrições longas são fatiadas em janelas e o score de cada doença é o máximo entre janelas.

## 1. Instalação

In [ ]:
# Mesmas versões devem ir para o requirements.txt do backend.
!pip install -q fastembed==0.8.1 faiss-cpu==1.15.1

## 2. Configuração

In [ ]:
import hashlib, json, re, shutil, unicodedata, zipfile
from collections import Counter
from pathlib import Path

import faiss
import numpy as np

# Modelo de embedding — trocar qualquer um destes invalida o índice (rebuild completo).
MODEL_NAME = "Xenova/multilingual-e5-small"
MODEL_FILE = "onnx/model_quantized.onnx"   # int8, 118 MB
DIM = 384

# Chunking
CHUNK_CHARS = 1200      # ~300 tokens, cabe folgado nos 512 do e5
OVERLAP = 200
MIN_CHUNK_CHARS = 80    # descarta sobras tipo "Chapter 4.1."
QUERY_WINDOW_CHARS = 1500

# Seções descartadas (só a própria seção; a próxima "## " volta a valer)
SECAO_DESCARTE = re.compile(
    r"^(\d+\.\s*)?(REFERENCES|SELECTED REFERENCES|FURTHER READING|APPENDIX|TABLE OF CONTENTS|FIGURES$|TABLES$"
    r"|ISBN|Disclaimer|ESCCAP believes|To achieve these objectives)",
    re.IGNORECASE,
)
# Vacinas (WOAH): descarta tudo até a próxima seção de topo "A." / "SUMMARY"
SECAO_VACINA = re.compile(r"^[A-Z]\.\s+REQUIREMENTS FOR VACCINES", re.IGNORECASE)
SECAO_TOPO = re.compile(r"^([A-Z]\.\s|SUMMARY\b)")
# Linha de bibliografia ("Sobrenome AB,", "- SOBRENOME A.B.", "1. Sobrenome, A") — exige ano na linha.
# Pega listas de referência que o Docling quebrou em várias "## " (cabeçalho de página, DOI).
LINHA_REF = re.compile(r"^\s*([-*•]\s*|\d+\.?\s*)?[A-Z][\w'’\-]+(\s[a-z]{1,3})?(\s[A-Z][\w'’\-]+)?,?\s+([A-Z]\.?\s?){1,3}[,.(:]")
ANO = re.compile(r"\b(19|20)\d{2}\b")
# Linhas-ruído: imagens, tabelas markdown, créditos de foto
LINHA_RUIDO = re.compile(r"^\s*(<!--.*-->|\|.*|(## )?Photo from .*)\s*$")

# Espécie (NM_ESPECIE do Oracle, PT) -> categorias dos documentos.
# Match por substring no nome normalizado (minúsculo, sem acento).
# Categorias sempre incluídas: doenças multiespécie.
CATEGORIAS_SEMPRE = {"Multiple Species", "Other Diseases"}
MAPA_ESPECIE = {
    "bovin": {"Bovinae"}, "vaca": {"Bovinae"}, "bufal": {"Bovinae"},
    "ovin": {"Caprinae"}, "ovelha": {"Caprinae"}, "caprin": {"Caprinae"}, "cabra": {"Caprinae"},
    "equin": {"Equidae"}, "cavalo": {"Equidae"}, "egua": {"Equidae"}, "asin": {"Equidae"}, "mul": {"Equidae"},
    "suin": {"Suidae"}, "porc": {"Suidae"},
    "ave": {"Aves"}, "galinha": {"Aves"}, "frango": {"Aves"}, "canari": {"Aves"}, "pato": {"Aves"}, "peru": {"Aves"},
    "coelho": {"Leporidae"}, "lebre": {"Leporidae"},
    "camel": {"Camelidae"}, "dromedario": {"Camelidae"},
    "abelha": {"Apinae"},
    "peixe": {"Fish"}, "tilapia": {"Fish"}, "salmao": {"Fish"}, "carpa": {"Fish"},
    "camarao": {"Crustaceans"}, "crustace": {"Crustaceans"}, "lagost": {"Crustaceans"},
    "ostra": {"Molluscs"}, "molusc": {"Molluscs"}, "mexilhao": {"Molluscs"},
    "anfibi": {"Amphibians"}, "sapo": {"Amphibians"}, "ra": {"Amphibians"},
    "canin": {"Canidae"}, "cao": {"Canidae"}, "cachorro": {"Canidae"},
    "felin": {"Felidae"}, "gato": {"Felidae"},
    "cervo": {"Cervidae"}, "veado": {"Cervidae"}, "cervid": {"Cervidae"},
    "morcego": {"Chiroptera"},
    "elefant": {"Elephantidae"},
    "canguru": {"Macropodidae"}, "walab": {"Macropodidae"},
    "golfinho": {"Marine Mammals"}, "foca": {"Marine Mammals"}, "baleia": {"Marine Mammals"},
    "furao": {"Mustelidae"}, "mustel": {"Mustelidae"},
    "primata": {"Primates"}, "macaco": {"Primates"}, "sagui": {"Primates"},
    "reptil": {"Reptilia"}, "serpente": {"Reptilia"}, "cobra": {"Reptilia"}, "lagarto": {"Reptilia"},
    "jabuti": {"Reptilia"}, "tartaruga": {"Reptilia"}, "iguana": {"Reptilia"},
    "roedor": {"Rodentia"}, "hamster": {"Rodentia"}, "rato": {"Rodentia"}, "camundongo": {"Rodentia"},
    "chinchila": {"Rodentia"}, "porquinho": {"Rodentia"},
}

INDEX_DIR = Path("/content/index")
DOCS_DIR = Path("/content/docs")

## 3. Upload (`docs.zip` obrigatório, `rag_index.zip` opcional)

In [ ]:
from google.colab import files

shutil.rmtree(INDEX_DIR, ignore_errors=True)
shutil.rmtree(DOCS_DIR, ignore_errors=True)
INDEX_DIR.mkdir(parents=True)
DOCS_DIR.mkdir(parents=True)

enviados = files.upload()
for nome in enviados:
    destino = INDEX_DIR if "index" in nome.lower() else DOCS_DIR
    with zipfile.ZipFile(nome) as z:
        z.extractall(destino)
    print(f"{nome} -> {destino}")

# Índice pode vir com ou sem pasta raiz dentro do zip
achado = next(INDEX_DIR.rglob("manifest.json"), None)
if achado and achado.parent != INDEX_DIR:
    for f in achado.parent.iterdir():
        shutil.move(str(f), INDEX_DIR / f.name)

print("Índice existente:", (INDEX_DIR / "manifest.json").exists())
print("Arquivos .md recebidos:", len(list(DOCS_DIR.rglob("*.md"))))

## 4. Modelo de embedding

In [ ]:
from fastembed import TextEmbedding
from fastembed.common.model_description import ModelSource, PoolingType

TextEmbedding.add_custom_model(
    model=MODEL_NAME,
    pooling=PoolingType.MEAN,
    normalization=True,
    sources=ModelSource(hf=MODEL_NAME),
    dim=DIM,
    model_file=MODEL_FILE,
)
modelo = TextEmbedding(model_name=MODEL_NAME)

def embed(textos, batch_size=64):
    """Vetores float32 normalizados (produto interno = cosseno)."""
    return np.array(list(modelo.embed(textos, batch_size=batch_size)), dtype="float32")

print(embed(["query: teste"]).shape)

## 5. Carregar ou criar índice

In [ ]:
CONFIG_INDICE = {"model": MODEL_NAME, "model_file": MODEL_FILE, "dim": DIM,
                 "chunk_chars": CHUNK_CHARS, "overlap": OVERLAP}

if (INDEX_DIR / "manifest.json").exists():
    manifest = json.loads((INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
    divergente = {k: (manifest["config"].get(k), v) for k, v in CONFIG_INDICE.items()
                  if manifest["config"].get(k) != v}
    if divergente:
        raise RuntimeError(f"Índice gerado com outra configuração {divergente}. "
                           "Rode sem rag_index.zip para reconstruir do zero.")
    index = faiss.read_index(str(INDEX_DIR / "faiss.index"))
    with open(INDEX_DIR / "chunks.jsonl", encoding="utf-8") as f:
        chunks = {c["id"]: c for c in map(json.loads, f)}
else:
    manifest = {"config": CONFIG_INDICE, "next_id": 0, "files": {}}
    index = faiss.IndexIDMap2(faiss.IndexFlatIP(DIM))
    chunks = {}

print(f"Vetores: {index.ntotal} | chunks: {len(chunks)} | documentos: {len(manifest['files'])}")

## 6. Leitura, limpeza e chunking dos `.md`

In [ ]:
def ler_frontmatter(texto):
    """Retorna (metadados, corpo). Frontmatter simples `chave: "valor"` entre `---`."""
    meta = {}
    if texto.startswith("---"):
        fim = texto.find("\n---", 3)
        if fim != -1:
            for linha in texto[3:fim].splitlines():
                if ":" in linha:
                    k, v = linha.split(":", 1)
                    meta[k.strip()] = v.strip().strip('"')
            texto = texto[fim + 4:]
    return meta, texto


def parece_referencias(texto):
    """Seção cuja maioria das linhas é entrada de bibliografia."""
    linhas = [l for l in texto.splitlines() if l.strip()]
    refs = sum(bool(LINHA_REF.match(l) and ANO.search(l)) or "doi.org" in l.lower() for l in linhas)
    return bool(linhas) and refs >= 0.5 * len(linhas)


def secoes_limpas(corpo):
    """Gera (titulo_secao, texto) pulando seções-ruído e linhas-ruído. Cada seção é avaliada sozinha."""
    secao, linhas, vacina = "", [], False
    for linha in corpo.splitlines() + ["## "]:          # sentinela fecha a última seção
        if LINHA_RUIDO.match(linha):
            continue
        if linha.startswith("## "):
            texto = "\n".join(linhas)
            if linhas and not vacina and not SECAO_DESCARTE.match(secao) and not parece_referencias(texto):
                yield secao, texto
            secao, linhas = linha[3:].strip(), []
            if SECAO_VACINA.match(secao):
                vacina = True
            elif SECAO_TOPO.match(secao):
                vacina = False
            continue
        linhas.append(linha)


def fatiar(texto, tam=CHUNK_CHARS, overlap=OVERLAP):
    """Janelas de ~tam chars, cortando em fim de frase quando possível."""
    texto = re.sub(r"\s+", " ", texto).strip()
    out, i = [], 0
    while i < len(texto):
        fim = min(i + tam, len(texto))
        if fim < len(texto):
            corte = texto.rfind(". ", i + tam // 2, fim)
            if corte != -1:
                fim = corte + 1
        out.append(texto[i:fim].strip())
        if fim >= len(texto):
            break
        prox = texto.find(" ", fim - overlap)
        i = prox + 1 if i < prox < fim else fim
    return out


def chunks_do_arquivo(caminho, raiz):
    """Lista de dicts (sem id) de um .md. Metadados do frontmatter, com fallback pelo caminho."""
    rel = caminho.relative_to(raiz)
    meta, corpo = ler_frontmatter(caminho.read_text(encoding="utf-8"))
    base = {
        "arquivo": rel.as_posix(),
        "titulo": meta.get("titulo") or caminho.stem,
        "manual": meta.get("manual") or (rel.parts[0] if len(rel.parts) > 1 else ""),
        "categoria": meta.get("categoria") or (rel.parts[-2] if len(rel.parts) > 1 else ""),
        "fonte": meta.get("fonte", ""),
    }
    out = []
    for secao, texto in secoes_limpas(corpo):
        for pedaco in fatiar(texto):
            if len(pedaco) >= MIN_CHUNK_CHARS:
                out.append({**base, "secao": secao, "ordem": len(out), "texto": pedaco})
    return out


def texto_para_embed(c):
    return f"passage: {c['titulo']} — {c['secao']}\n{c['texto']}"

## 7. Detectar arquivos novos / alterados

In [ ]:
raiz = DOCS_DIR
novos, alterados, iguais = [], [], []
for caminho in sorted(raiz.rglob("*.md")):
    if "__MACOSX" in caminho.parts:
        continue
    rel = caminho.relative_to(raiz).as_posix()
    sha = hashlib.sha256(caminho.read_bytes()).hexdigest()
    antigo = manifest["files"].get(rel)
    if antigo is None:
        novos.append((caminho, sha))
    elif antigo["sha256"] != sha:
        alterados.append((caminho, sha))
    else:
        iguais.append(rel)

# Mesmo nome de arquivo já indexado sob outro caminho = zip montado com outra pasta raiz.
# Seguir criaria duplicatas no índice.
indexados = {Path(r).name: r for r in manifest["files"]}
suspeitos = [(c.relative_to(raiz).as_posix(), indexados[c.name]) for c, _ in novos if c.name in indexados]
if suspeitos:
    raise RuntimeError("Arquivos já indexados com outro caminho (zip com pasta raiz diferente?):\n"
                       + "\n".join(f"  novo: {a}\n  índice: {b}" for a, b in suspeitos[:5]))

print("Exemplo de chave:", (novos + alterados)[0][0].relative_to(raiz).as_posix() if novos + alterados else "-")
print(f"Novos: {len(novos)} | alterados: {len(alterados)} | já indexados (ignorados): {len(iguais)}")

## 8. Vetorizar e adicionar ao índice

In [ ]:
# Alterados: remove chunks antigos antes de reindexar
for caminho, _ in alterados:
    rel = caminho.relative_to(raiz).as_posix()
    ids_antigos = manifest["files"][rel]["ids"]
    index.remove_ids(np.array(ids_antigos, dtype="int64"))
    for i in ids_antigos:
        chunks.pop(i, None)

for n, (caminho, sha) in enumerate(novos + alterados, 1):
    rel = caminho.relative_to(raiz).as_posix()
    novos_chunks = chunks_do_arquivo(caminho, raiz)
    ids = list(range(manifest["next_id"], manifest["next_id"] + len(novos_chunks)))
    manifest["next_id"] += len(novos_chunks)
    if novos_chunks:
        vetores = embed([texto_para_embed(c) for c in novos_chunks])
        index.add_with_ids(vetores, np.array(ids, dtype="int64"))
        for i, c in zip(ids, novos_chunks):
            chunks[i] = {"id": i, **c}
    manifest["files"][rel] = {"sha256": sha, "ids": ids}
    print(f"[{n}/{len(novos) + len(alterados)}] {rel}: {len(novos_chunks)} chunks")

assert index.ntotal == len(chunks), (index.ntotal, len(chunks))
print(f"Total no índice: {index.ntotal} vetores")

## 9. Salvar e baixar `rag_index.zip`

In [ ]:
faiss.write_index(index, str(INDEX_DIR / "faiss.index"))
with open(INDEX_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for i in sorted(chunks):
        f.write(json.dumps(chunks[i], ensure_ascii=False) + "\n")
(INDEX_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")

with zipfile.ZipFile("rag_index.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for nome in ("faiss.index", "chunks.jsonl", "manifest.json"):
        z.write(INDEX_DIR / nome, nome)
files.download("rag_index.zip")

## 10. Estatísticas

In [ ]:
docs = {c["arquivo"] for c in chunks.values()}
print(f"Documentos: {len(docs)} | chunks: {len(chunks)}")
for nome in ("faiss.index", "chunks.jsonl", "manifest.json"):
    print(f"  {nome}: {(INDEX_DIR / nome).stat().st_size / 1e6:.1f} MB")
print("\nChunks por categoria:")
for cat, n in Counter(c["categoria"] for c in chunks.values()).most_common():
    print(f"  {cat or '(sem categoria)'}: {n}")

## 11. Busca (mesma lógica que irá para o backend)

`buscar(texto, especie)` → top-3 documentos distintos (por fonte + título). Score do documento = maior similaridade entre qualquer chunk dele e qualquer janela da transcrição.

In [ ]:
def _norm(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()


def categorias_da_especie(especie):
    """None = sem filtro (espécie vazia ou desconhecida)."""
    if not especie:
        return None
    palavras = re.findall(r"[a-z]+", _norm(especie))
    cats, achou = set(CATEGORIAS_SEMPRE), False
    for chave, valor in MAPA_ESPECIE.items():
        # chaves curtas ("ra", "ave", "can") exigem palavra inteira/prefixo, não substring solta
        if any(p == chave or (len(chave) >= 3 and p.startswith(chave)) for p in palavras):
            cats |= valor
            achou = True
    return cats if achou else None


def buscar(texto, especie=None, k=3, k_chunks=100):
    janelas = fatiar(texto, tam=QUERY_WINDOW_CHARS) or [texto]
    q = embed([f"query: {j}" for j in janelas])
    scores, ids = index.search(q, min(k_chunks, index.ntotal))
    permitidas = categorias_da_especie(especie)

    melhor_por_chunk = {}
    for linha_s, linha_i in zip(scores, ids):
        for s, i in zip(linha_s, linha_i):
            if i != -1 and s > melhor_por_chunk.get(i, -1):
                melhor_por_chunk[int(i)] = float(s)

    doencas = {}
    for i, s in sorted(melhor_por_chunk.items(), key=lambda x: -x[1]):
        c = chunks[i]
        if permitidas is not None and c["categoria"] not in permitidas:
            continue
        # (fonte, titulo): guias ESCCAP repetidos em Canidae/Felidae contam uma vez só
        d = doencas.setdefault((c["fonte"], c["titulo"]), {"titulo": c["titulo"], "categoria": c["categoria"],
                                                            "fonte": c["fonte"], "score": s, "trechos": []})
        if len(d["trechos"]) < 2:
            d["trechos"].append(c)

    top = sorted(doencas.values(), key=lambda d: -d["score"])[:k]
    for d in top:  # resumo = primeiro chunk do documento (SUMMARY / Scope)
        arquivo = d["trechos"][0]["arquivo"]
        d["resumo"] = next(c for c in chunks.values() if c["arquivo"] == arquivo and c["ordem"] == 0)
        d["trechos"] = [c for c in d["trechos"] if c["id"] != d["resumo"]["id"]]
    return top


def montar_contexto_rag(top, max_chars=3000):
    """Bloco de texto para o prompt do Groq. Orçamento dividido igualmente entre as doenças."""
    if not top:
        return ""
    por_doenca = max_chars // len(top)
    partes = ["BASE DE CONHECIMENTO VETERINÁRIA (RAG — WOAH, AAZV, CFSPH, CAPC, ABCD, ESCCAP; em inglês). "
              "Documentos mais relevantes para o relato (fichas de doença ou guias clínicos), "
              "por similaridade semântica (não é probabilidade):"]
    for n, d in enumerate(top, 1):
        cab = f"\n[{n}] {d['titulo']} ({d['categoria']}, fonte: {d['fonte'] or 'n/d'}, relevância {d['score']:.2f})\n"
        corpo = " […] ".join([d["resumo"]["texto"]] + [f"({c['secao']}) {c['texto']}" for c in d["trechos"]])
        partes.append(cab + corpo[: por_doenca - len(cab)])
    return "\n".join(partes)

## 12. Testes com relatos em PT-BR

In [ ]:
EXEMPLOS = [
    ("Bovino", "Vaca leiteira com queda brusca na produção de leite, mucosas pálidas e amareladas, "
               "fraqueza, perda de peso e muitos carrapatos no rebanho. Sem urina escura."),
    ("Ave", "Galinhas com espirros, secreção nasal, dificuldade respiratória, queda na postura "
            "de ovos com casca fina e deformada. Alta mortalidade em pintinhos."),
    ("Canino", "Cão com mudança súbita de comportamento, agressividade, salivação excessiva, "
               "dificuldade para engolir e paralisia progressiva das patas traseiras. Mordido por morcego."),
    ("Felino", "Gato com perda de peso, vômito, diarreia crônica e pelagem opaca. "
               "Vive com outros gatos e caça ratos."),
    ("", "Animal com febre, aborto no terço final da gestação e retenção de placenta."),
]

for especie, relato in EXEMPLOS:
    print(f"\n=== Espécie: {especie or '(sem filtro)'} ===\n{relato}")
    for d in buscar(relato, especie):
        print(f"  {d['score']:.3f}  {d['titulo']}  [{d['categoria']}]")

print("\n--- Prévia do bloco que irá para o LLM (exemplo 1) ---")
print(montar_contexto_rag(buscar(EXEMPLOS[0][1], EXEMPLOS[0][0])))